# CSA adoption rates and logistic regression

Produces Table 4.11 and Table 4.12.

Personal file paths, stored outputs, exploratory analyses, and superseded cells have been removed.

## Reproducibility note

The uploaded result-producing cell used a four-item lower-bound empowerment proxy:
decision-making, credit access, extension access, and a land-access proxy. This implementation
reproduced the values reported in Tables 4.11 and 4.12. The thesis methods describe a
five-item formulation, so the code–methods wording should be reconciled before archival
publication.

In [ ]:
from pathlib import Path
import warnings

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name in {"notebooks", "needs_verification"}:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_FILE = PROJECT_DIR / "data" / "questionnaire.xlsx"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {DATA_FILE}\n"
        "Place the anonymised questionnaire file in data/questionnaire.xlsx "
        "or update DATA_FILE."
    )

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

OUT_DIR = OUTPUTS_DIR / "02_csa_adoption"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_FILE)
df.columns = df.columns.str.strip()

adaptation_col = "Have you implemented at least one adaptation strategy?"
training_col = "If yes, how many training programs have you attended?"
farm_size_col = "What is the size of your farm (in acres)?"

required = [
    "Location",
    adaptation_col,
    training_col,
    farm_size_col,
    "Are women involved in decision-making processes related to crop selection, marketing, and income management?",
    "Do you face any gender-related challenges in accessing agricultural resources (e.g., land, credit, inputs)?",
    "Do women have equal access to agricultural training and extension services?",
    "Do you own or have access to agricultural land?",
]
missing = [column for column in required if column not in df.columns]
if missing:
    raise KeyError(f"Missing questionnaire columns:\n{missing}")

df["adapt_uptake"] = (
    df[adaptation_col].astype("string").str.strip().str.lower().eq("yes").astype(int)
)

df["train_5plus"] = (
    df[training_col].astype("string").str.strip().str.lower()
      .map({"1-5": 0, "5-10": 1, ">10": 1, "more than 10": 1})
      .fillna(0)
      .astype(int)
)

df["scheme_perkerra"] = (
    df["Location"].astype("string").str.contains("Perkerra", case=False, na=False).astype(int)
)

df["farm_size_acres"] = (
    df[farm_size_col].astype("string")
      .str.replace(",", "", regex=False)
      .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
      .pipe(pd.to_numeric, errors="coerce")
)
df["farm_size_ha"] = df["farm_size_acres"] * 0.40468564224

# Lower-bound empowerment proxy used in the retained result-producing code.
df["weai_decide"] = (
    df["Are women involved in decision-making processes related to crop selection, marketing, and income management?"]
      .astype("string").str.strip().str.lower().eq("yes").astype(int)
)
df["weai_credit"] = (
    df["Do you face any gender-related challenges in accessing agricultural resources (e.g., land, credit, inputs)?"]
      .astype("string").str.strip().str.lower().eq("no").astype(int)
)
df["weai_extension"] = (
    df["Do women have equal access to agricultural training and extension services?"]
      .astype("string").str.strip().str.lower().eq("yes").astype(int)
)
df["weai_equipment"] = (
    df["Do you own or have access to agricultural land?"]
      .astype("string").str.strip().str.lower().eq("yes").astype(int)
)

weai_items = ["weai_decide", "weai_credit", "weai_extension", "weai_equipment"]
df["sWEAI_score"] = df[weai_items].mean(axis=1)
median_score = df["sWEAI_score"].median()
df["sWEAI_group"] = np.where(df["sWEAI_score"] > median_score, "High", "Low")


In [ ]:
# Table 4.11: adoption rates
def adoption_rate(mask=None):
    values = df["adapt_uptake"] if mask is None else df.loc[mask, "adapt_uptake"]
    return round(values.mean() * 100, 1)

table_411 = pd.DataFrame([
    {"Group": "Overall sample", "Adoption_rate_percent": adoption_rate()},
    {"Group": "TIS", "Adoption_rate_percent": adoption_rate(df["scheme_perkerra"] == 0)},
    {"Group": "PIS", "Adoption_rate_percent": adoption_rate(df["scheme_perkerra"] == 1)},
    {"Group": "<5 trainings", "Adoption_rate_percent": adoption_rate(df["train_5plus"] == 0)},
    {"Group": "≥5 trainings", "Adoption_rate_percent": adoption_rate(df["train_5plus"] == 1)},
    {"Group": "Low empowerment", "Adoption_rate_percent": adoption_rate(df["sWEAI_group"] == "Low")},
    {"Group": "High empowerment", "Adoption_rate_percent": adoption_rate(df["sWEAI_group"] == "High")},
])

table_411.to_excel(OUT_DIR / "table_4_11_adoption_rates.xlsx", index=False)
display(table_411)


In [ ]:
# Table 4.12: logistic regression predicting CSA adoption
model_columns = [
    "scheme_perkerra", "train_5plus", "sWEAI_score",
    "farm_size_ha", "adapt_uptake"
]
model_data = df[model_columns].dropna().copy()

X = sm.add_constant(
    model_data[["scheme_perkerra", "train_5plus", "sWEAI_score", "farm_size_ha"]],
    has_constant="add",
)
y = model_data["adapt_uptake"]

model = sm.Logit(y, X).fit(disp=False)
ci = model.conf_int()

table_412 = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "StdErr": model.bse.values,
    "p_value": model.pvalues.values,
    "Odds_Ratio": np.exp(model.params.values),
    "CI_Low": np.exp(ci[0].values),
    "CI_High": np.exp(ci[1].values),
})

table_412["Variable"] = table_412["Variable"].replace({
    "const": "Intercept",
    "scheme_perkerra": "PIS",
    "train_5plus": "≥5 trainings",
    "sWEAI_score": "sWEAI score",
    "farm_size_ha": "Farm size (ha)",
})

numeric_columns = [
    "Coefficient", "StdErr", "p_value", "Odds_Ratio", "CI_Low", "CI_High"
]
table_412[numeric_columns] = table_412[numeric_columns].round(3)

table_412.to_excel(OUT_DIR / "table_4_12_csa_adoption_logit.xlsx", index=False)
display(table_412)
print(f"Model N = {int(model.nobs)}")
